# Section 3 — Quantize a model and measure the trade-off

**Model:** Qwen3.5-4B (dense 4.21B, Apache 2.0) · **Hardware:** MacBook Pro M5,
16 GB unified memory, **on AC power** · **Runtime:** llama.cpp via Metal
(`brew install llama.cpp`)

> Power-state note (measured): on battery, Apple Silicon throttles the GPU —
> the same Q4 bench that reports ~35 t/s on AC dropped to ~18-24 t/s on
> battery. All numbers below were produced plugged in.

This notebook IS the pipeline — each step below is executed for real, in
order, and every measurement is also written to `results/` as a raw
artifact. The blind judge lives in [judge.py](judge.py) (imported in Step 8).

| Step | What happens |
|---|---|
| 1 | Get the BF16 (full-precision) model |
| 2 | **Quantize it ourselves** → Q4_K_M |
| 3 | Measure memory footprint, both versions |
| 4 | Measure tokens/sec, both versions |
| 5 | Same 5 prompts through both versions |
| 6 | Trade-off table + conclusions |

## Step 1 — the full-precision baseline

BF16 GGUF = the original 16-bit weights, repackaged for llama.cpp. ~8.4 GB,
so the cell skips the download if the file is already present.

In [1]:
import os, re, ssl, subprocess, urllib.request
from pathlib import Path
import certifi

Path("results").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

URL = "https://huggingface.co/unsloth/Qwen3.5-4B-GGUF/resolve/main/Qwen3.5-4B-BF16.gguf"
bf16 = Path("models/Qwen3.5-4B-BF16.gguf")
FULL = 8424393632  # exact size from the HF API

if not bf16.exists() or bf16.stat().st_size < FULL:
    # this Python install has no CA certs wired in -- use certifi's bundle
    ctx = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(URL, context=ctx) as r, open(bf16, "wb") as f:
        while chunk := r.read(1 << 20):
            f.write(chunk)
print(f"BF16 model: {bf16.stat().st_size / 1024**3:.2f} GiB")

MODELS = ["Qwen3.5-4B-BF16", "Qwen3.5-4B-Q4_K_M"]  # used by every step below

BF16 model: 7.85 GiB


## Step 2 — quantize it (the heart of this section)

`llama-quantize` converts BF16 → **Q4_K_M**: most weights become 4-bit blocks
with per-block scale factors; the most damage-sensitive tensors (attention,
output head) are kept at higher precision. This cell performs the actual
quantization, live:

In [2]:
out = subprocess.run(
    ["llama-quantize", str(bf16), "models/Qwen3.5-4B-Q4_K_M.gguf", "Q4_K_M"],
    capture_output=True, text=True)
print("\n".join(out.stdout.splitlines()[-4:]))
q4 = Path("models/Qwen3.5-4B-Q4_K_M.gguf")
print(f"\nBF16: {bf16.stat().st_size/1024**3:.2f} GiB  ->  Q4_K_M: {q4.stat().st_size/1024**3:.2f} GiB "
      f"({q4.stat().st_size/bf16.stat().st_size:.0%} of original)")


llama_quantize: quantize time = 23038.67 ms
llama_quantize:    total time = 23038.67 ms

BF16: 7.85 GiB  ->  Q4_K_M: 2.52 GiB (32% of original)


## Step 3 — memory footprint

We load each model with an identical, pinned 8192-token context and read
**llama.cpp's own allocation log** — the honest number. (macOS `time -l`
peak-RSS is misleading here: under Metal both models report the ~11.4 GiB
unified-memory working-set ceiling.)

**Memory hygiene:** every measurement in this notebook runs the model as a
separate OS process that fully exits before the next starts — so BF16 and
Q4 are *never* resident at the same time (16 GB could not hold both plus a
KV cache).

In [3]:
import pandas as pd

def measure_ram(name):
    p = subprocess.run(
        ["llama-cli", "-m", f"models/{name}.gguf", "-st", "--temp", "0", "-n", "4",
         "--reasoning", "off", "-c", "8192", "-v", "-p", "hi"],
        capture_output=True, text=True)
    lines = (p.stdout + p.stderr).splitlines()
    start = next(i for i, l in enumerate(lines) if "_Mapped model buffer size" in l)
    sizes = {}
    for line in lines[start:]:
        m = re.search(r"(\S+(?:_Mapped model| KV| RS| compute)) buffer size =\s+([\d.]+) MiB", line)
        if m and m.group(1) not in sizes:
            sizes[m.group(1)] = float(m.group(2))
    Path(f"results/ram_{name}.txt").write_text(
        "".join(f"{k} buffer size = {v} MiB\n" for k, v in sizes.items()))
    return sizes

rows = {}
for name in MODELS:  # sequential: each subprocess exits before the next runs
    s = measure_ram(name)
    weights = sum(v for k, v in s.items() if "_Mapped model" in k)
    other = sum(v for k, v in s.items() if "_Mapped model" not in k)
    rows[name] = {"weights (MiB)": round(weights), "KV+state+compute (MiB)": round(other),
                  "total (GiB)": round((weights + other) / 1024, 2)}
ram = pd.DataFrame(rows).T
ram["vs BF16"] = (ram["total (GiB)"] / ram.loc[MODELS[0], "total (GiB)"] - 1).map("{:+.0%}".format)
ram

,weights (MiB),KV+state+compute (MiB),total (GiB),vs BF16
Qwen3.5-4B-BF16,9236.0,412.0,9.42,+0%
Qwen3.5-4B-Q4_K_M,3070.0,412.0,3.40,-64%


**Why does "weights in memory" read larger than the 7.84 GiB file?**
Two reasons, one real and one bookkeeping:

- *Real:* inference adds a KV cache + compute buffers on top of the weights
  (the 412 MiB row) — memory beyond the file is expected.
- *Bookkeeping:* the sum above double-counts the **tied token-embedding
  tensor** (248320 vocab × 2560 dims = 1212 MiB in BF16, 497 MiB in Q4):
  it is mapped by both the CPU backend (input lookup) and Metal (output
  head) — two views of the same mmap-shared physical pages. The **unique**
  footprint is ≈ file size + 412 MiB: **~8.24 GiB (BF16) vs ~2.92 GiB
  (Q4_K_M), −65%.**

## Step 4 — throughput (tokens/sec)

`llama-bench`, 5 repetitions each: `pp512` = prompt processing,
`tg128` = token generation. Takes a few minutes (BF16 generation is slow —
that is the point).

In [4]:
def bench(name):
    p = subprocess.run(["llama-bench", "-m", f"models/{name}.gguf", "-p", "512", "-n", "128"],
                       capture_output=True, text=True)
    Path(f"results/bench_{name}.txt").write_text(p.stdout)
    rows = {}
    for line in p.stdout.splitlines():
        m = re.search(r"\|\s+(pp512|tg128)\s+\|\s+([\d.]+)", line)
        if m:
            rows[m.group(1)] = float(m.group(2))
    return rows

speed = pd.DataFrame({n: bench(n) for n in MODELS}).T
speed.columns = ["prompt processing (t/s)", "generation (t/s)"]
speed["gen speedup"] = (speed["generation (t/s)"] / speed.loc[MODELS[0], "generation (t/s)"]).map("{:.1f}x".format)
speed

,prompt processing (t/s),generation (t/s),gen speedup
Qwen3.5-4B-BF16,954.58,13.03,1.0x
Qwen3.5-4B-Q4_K_M,1013.48,35.04,2.7x


**Why generation speeds up ~2.7× but prompt processing doesn't:** generation
streams *all* weights through the memory bus for *every* token — it is
bandwidth-bound, and 4-bit weights are ~⅓ the bytes. Prompt processing is
batched matmuls (compute-bound); dequantization overhead roughly cancels the
bandwidth gain. This asymmetry is the entire speed story of quantization.

## Step 5 — quality: 5 fixed prompts, both versions

Greedy decoding (`--temp 0`) so outputs are deterministic — any difference is
the quantization. Reasoning pinned off (Qwen3.5 is a hybrid-thinking model;
variable thinking traces would make outputs incomparable). Prompts probe:
factual+constraint, exact math, code, JSON extraction, and Egyptian-Arabic
dialect — deliberately the model's weakest skill.

In [5]:
def generate(name, prompt_file):
    p = subprocess.run(
        ["llama-cli", "-m", f"models/{name}.gguf", "-f", str(prompt_file), "-st",
         "--temp", "0", "-n", "640", "--reasoning", "off", "--no-display-prompt"],
        capture_output=True, text=True)
    out_dir = Path(f"results/quality_{name}")
    out_dir.mkdir(exist_ok=True)
    (out_dir / prompt_file.name).write_text(p.stdout)
    m = re.search(r"^> .*?$(.*?)\[ Prompt:", p.stdout, re.S | re.M)
    return (m.group(1).strip() if m else p.stdout.strip())

answers = {}
for pf in sorted(Path("prompts").glob("*.txt")):
    answers[pf.name] = {n: generate(n, pf) for n in MODELS}
    print("done:", pf.name)

done: 1_factual.txt


done: 2_math.txt


done: 3_code.txt


done: 4_extraction.txt


done: 5_arabic.txt


In [6]:
for pname, by_model in answers.items():
    print("=" * 78)
    print("PROMPT:", Path("prompts", pname).read_text().strip()[:110])
    for n in MODELS:
        print(f"\n--- {n}:\n{by_model[n]}")
    print()

PROMPT: Why does the Earth have seasons? Answer in exactly 3 sentences.

--- Qwen3.5-4B-BF16:
The Earth has seasons because its axis is tilted at an angle of approximately 23.5 degrees relative to its orbital plane. As the Earth orbits the Sun, this tilt causes different hemispheres to receive varying amounts of direct sunlight throughout the year. When a hemisphere is tilted toward the Sun, it experiences summer with longer days and more intense heat, while the opposite hemisphere experiences winter.

--- Qwen3.5-4B-Q4_K_M:
The Earth has seasons because its axis is tilted at an angle of approximately 23.5 degrees relative to its orbital plane. As the Earth orbits the Sun, this tilt causes different hemispheres to receive varying amounts of direct sunlight throughout the year. When a hemisphere tilts toward the Sun, it experiences summer due to more direct and intense solar radiation, while the hemisphere tilted away experiences winter with less direct light.

PROMPT: A restaurant sell

## Step 6 — the trade-off, summarized

All numbers below are from this notebook's own execution (AC power).
Absolute throughput varies with machine load between runs; the BF16:Q4
*ratio* is stable (2.5–2.9× in every condition measured).

| | BF16 | Q4_K_M | change |
|---|---|---|---|
| Precision | 16-bit | ~5.1 bit effective | — |
| Weights in RAM (= file size) | 7.84 GiB | 2.51 GiB | **−68%** |
| + KV cache + state + compute (8k ctx) | 0.40 GiB | 0.40 GiB | same |
| **= Total inference RAM (Step 3)** | **8.24 GiB** | **2.92 GiB** | **−65%** |
| Generation speed (Step 4) | 13.0 t/s | 35.0 t/s | **2.7×** |
| Prompt processing (Step 4) | ~955 t/s | ~1013 t/s | ≈ same |
| TTFT, 732-token prompt (Step 7) | 958 ms | 800 ms | **−17%** |
| Quality — EN / math / code / JSON (Steps 5+8) | reference | indistinguishable | tie |
| Quality — Egyptian Arabic (Steps 5+8) | coherent | broken phrasing | **degraded** |

### Conclusions

1. **Q4_K_M is the obvious serving choice** for this model on this machine:
   a third of the memory, ~2.5–2.9× generation speed, no measurable loss on
   mainstream tasks. It is the file Section 4 deploys.
2. **The speed win lives in decode; TTFT improves far less** — 2.7× on
   generation but only ~17% on first-token latency, because TTFT is
   dominated by compute-bound prefill (Step 7).
3. **Quantization loss is not uniform — it erodes the tails first.** It
   didn't show in English, math, code, or JSON; it surfaced in the model's
   weakest capability (Egyptian dialect), confirmed blind by the judge
   (Step 8). Rule: evaluate a quantized model on your own weakest-case
   workload — standard benchmarks are where the loss hides.
4. Honest caveats are in the README; the production-format question
   (GPTQ/AWQ vs bitsandbytes vs GGUF) is answered in [NOTES.md](NOTES.md).

## Step 7 — time-to-first-token *(added beyond the task requirements)*

The spec asks for throughput only; TTFT is added because it is the metric a
*user actually feels* — the silence before the first word — and the one that
matters most for interactive serving (it is also a Section 4 requirement, so
it is baselined here).

TTFT = prefill (prompt eval) + the first decode step. One realistic
~730-token prompt, one request, each model in its own process (sequential —
never both in memory).

In [7]:
prompt = "Customer support context line about orders, refunds and delivery times. " * 60

def timing(name):
    p = subprocess.run(
        ["llama-cli", "-m", f"models/{name}.gguf", "-st", "--temp", "0", "-n", "32",
         "--reasoning", "off", "-v", "-p", prompt],
        capture_output=True, text=True)
    out = p.stdout + p.stderr
    Path(f"results/ttft_{name}.txt").write_text(
        "\n".join(l for l in out.splitlines() if "print_timing" in l) + "\n")
    pe = re.search(r"prompt eval time =\s+([\d.]+) ms /\s+(\d+) tokens", out)
    de = re.search(r"\|\s+eval time =\s+([\d.]+) ms /\s+(\d+) tokens \(\s*([\d.]+) ms per token", out)
    prefill_ms, n_prompt = float(pe.group(1)), int(pe.group(2))
    per_tok_ms = float(de.group(3))
    return {"prompt tokens": n_prompt, "prefill (ms)": round(prefill_ms, 1),
            "per-token decode (ms)": round(per_tok_ms, 1),
            "TTFT (ms)": round(prefill_ms + per_tok_ms, 1)}

ttft = pd.DataFrame({n: timing(n) for n in MODELS}).T
ttft

,prompt tokens,prefill (ms),per-token decode (ms),TTFT (ms)
Qwen3.5-4B-BF16,732.0,890.9,67.3,958.3
Qwen3.5-4B-Q4_K_M,732.0,773.6,26.0,799.6


## Step 8 — blind LLM-judge *(added beyond the task requirements)*

Rubric-based pairwise judging is the standard approach in modern model
evaluation and RLHF preference pipelines (MT-Bench / Arena style), and a
method I use by default: it turns "I eyeballed the outputs" into scored,
reproducible evidence.

Setup: `gemini-3.5-flash` (temp 0) scores both answers per prompt on the
standard rubrics — instruction following, truthfulness, conciseness,
writing style, helpfulness (1–5 each, with the issues it found) — then gives
a pairwise verdict on the 7-point scale (A much better … tie … B much
better).

Bias controls: the judge never sees model names, and the A/B assignment is
**randomized per prompt** (seeded → reproducible); identities are joined
back only after judging. Implementation: [judge.py](judge.py); raw output in
`results/judge_results.json`.

In [8]:
import importlib, json
import judge
importlib.reload(judge)
judge.main()

data = json.loads(Path("results/judge_results.json").read_text())
res = data["results"]

verdicts = pd.DataFrame([{"prompt": r["prompt"], "winner": r["winner"].split("-")[-1],
                          "margin": r["margin"], "why": r["why"]} for r in res])
display(verdicts)

avg = pd.DataFrame({m: {rub: pd.Series([r["scores"][m][rub] for r in res]).mean()
                        for rub in judge.RUBRICS} for m in judge.MODELS}).round(2)
avg

judging 1_factual.txt  (A=BF16)


judging 2_math.txt  (A=BF16)


judging 3_code.txt  (A=Q4_K_M)


judging 4_extraction.txt  (A=BF16)


judging 5_arabic.txt  (A=BF16)



wrote results/judge_results.json and results/judge_summary.md


,prompt,winner,margin,why
0,1_factual.txt,tie,tie,Both answers followed the negative constraint ...
1,2_math.txt,tie,tie,Both models followed all instructions perfectl...
2,3_code.txt,tie,tie,Both models implemented the correct merging al...
3,4_extraction.txt,tie,tie,Both models extracted the information perfectl...
4,5_arabic.txt,BF16,better,While Answer B followed the sentence count con...


,Qwen3.5-4B-BF16,Qwen3.5-4B-Q4_K_M
instruction_following,3.8,4.0
truthfulness,4.4,4.0
conciseness,4.8,4.6
writing_style,4.4,4.0
helpfulness,4.0,3.6
